<hr/>

# [Applied Linear Statistics Models](https://donnievin.github.io/ASDA1//)
By: **Donovan Vincent Jr** - dvincen9@jh.edu <br/>
Adapted From: **Dr. Sergey Kushnarev's ASDA I** <br/>
Estimated Workthrough Time: **90 mins**

<hr/>

<h1><font color="blue">Linear Regression with 1 predictor</font></h1>

Topics: 
* Simple Linear Regression (SLR)
* Brief Math Detour: Gradients, Jacobians, Hessians
* Ordinary Least Squares (OLS)
* Residuals: estimating Unexplained Variance
* Maximum Likelihood Estimation (MLE)
* Fisher Information Matrix

In [1]:
import time
import scipy
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import clear_output


# <font color='lightblue'> Simple Lienar Regression (SLR) </font>

> SLR Model

The summation operator gives us a compact way to represent long sums over an collection of variables and is described as follows:

<div align="center">

$$
Y_i = \beta_0 + \beta_1 X_i + \epsilon_i
$$

</div>

Where: 
* $Y_i$ = observed value of response (or outcome) variable
* $x_i$ = fixed <font color='red'>known</font> values of the predictor (or explanatory) variables
* $\beta_0, \beta_1$ = fixed <font color='red'>unknown</font> parameters 
* $\epsilon_i$ = unknown random variable (error) = soruce of noise in data

--- 


> Noise Assumptions $\epsilon_i$

This operator gives us a compact way to represent numerous multiplications over a collection of variables and is describes as follows: 



1. <font color='red'>Centered</font>: $\mathbb{E}[\epsilon_i] = 0$
2. <font color='red'>Homoskedasticity</font>: $Var[\epsilon_i] = \sigma^2$ (constant variance)
3. <font color='red'>Uncorrelated</font>: $Cov[\epsilon_i, \epsilon_j] = 0$


--- 

> Goal

Given data / observations, $\mathcal{D} = \{x_i, y_i\}_{i=1}^n$, find the "best" estimators for $\beta_0, \beta_1$, which we call $\hat{\beta_0}, \hat{\beta_1}$. To do so, we should minimize the Ordinary Least Squares


<div align="center">

$$
(\hat{\beta_0}, \hat{\beta_1}) = \argmin_{\beta_0, \beta_1} \sum_{i=1}^n (Y_i - \hat{Y_i})^2 = \argmin_{\beta_0, \beta_1} \sum_{i=1}^n (Y_i - \beta_0 - \beta_1 X_i )^2
$$

</div>


<font color='purple'> Note: Min returns the smallest output value (typically y), but Argmin returns the input value (argument) that would give us this value at which the function is minimized </font>



# <font color='lightblue'> Brief Math Detour: Gradients, Jacobians, Hessians </font>

> Types of functions

* Scalar valued function: $f: \mathbb{R}^n \to \mathbb{R}$ (e.g. Functionals, ... )
* Vector valued funtions: $F: \mathbb{R}^n \to \mathbb{R}^m$ (e.g. Matrix multiplication, ...)


--- 


> Gradients

The Gradient is the generalized form of the $1st$ derivative that quantifies: <font color='red'> The direction of steepest ascent </font>. 


If $f:  \mathbb{R}^n \to \mathbb{R}$ is differentiable, then we define the gradient of f at x to be:

<div align="center">

$$
{\nabla f(x) :=
\left(\begin{array}{c}
\frac{\partial f(x)}{\partial x_1}\\
\vdots \\
\frac{\partial f(x)}{\partial x_n}
\end{array}
\right)}
$$

</div>



---

> Hessians

The Hessian is the generalized form of the $2nd$ derivative that quantifies: <font color='red'> How curved is the function </font>. 


If $f:  \mathbb{R}^n \to \mathbb{R}$ is twice differentiable, then we define the hessian of f at x to be:


<div align="center">

$$
{\nabla^2 f(x) :=
\left(\begin{array}{ccc}
\frac{\partial^2 f(x)}{\partial x_1^2}   &   \dots  &  \frac{\partial^2 f(x)}{\partial x_n\partial x_1}  \\
\vdots   &   \ddots   &   \vdots \\
\frac{\partial^2 f(x)}{\partial x_1\partial x_n}  &  \dots   &  \frac{\partial^2 f(x)}{\partial x_n^2}
\end{array}
\right)}
$$

</div>


---

> Jacobian

The Jacobian is the generalized form of the Gradient that allows us to quantify: <font color='red'> How function values change with their inputs (How the coordinate system stretches/rotates) </font>. 


If $F:  \mathbb{R}^n \to \mathbb{R}^m$ is differentiable, then we define the Jacobian of F at x to be:


<div align="center">

$$
{J(x) := \nabla F(x) :=
\left(\begin{array}{ccc}
\frac{\partial F_1(x)}{\partial x_1}   &   \dots   &   \frac{\partial F_1(x)}{\partial x_n}  \\
\vdots   &   \ddots   &   \vdots \\
\frac{\partial F_m(x)}{\partial x_1}   &   \dots   &   \frac{\partial F_m(x)}{\partial x_n}
\end{array}
\right)
}$$ 

</div>

where ${F_i(x), \ i=1, \ldots, m}$ is the i-th component of $F(x)$.

---


> Relationships 

**FACT**: The <span style="color:purple">Hessian</span> is the <span style="color:purple">Jacobian</span> of the <span style="color:purple">Gradient</span> map.



# <font color='lightblue'> Ordinary Least Squares (OLS) </font>

> Normal Equations (SLR)

Our objective, or criterion, is to solve:

<div align="center">

$$
Q = \argmin_{\beta_0, \beta_1} \sum_{i=1}^n (Y_i - \beta_0 - \beta_1 X_i )^2
$$

</div>

1. $\frac{\partial Q}{\partial \beta_0} = \sum_{i=1}^n 2(Y_i - \beta_0 - \beta_1 X_i ) (-1) \implies \sum_{i=1}^n Y_i = n \hat{\beta_0} + \hat{\beta_1} \sum_{i=1}^n X_i$
2. $\frac{\partial Q}{\partial \beta_1} = \sum_{i=1}^n 2(Y_i - \beta_0 - \beta_1 X_i ) (-X_i) \implies \sum_{i=1}^n Y_i X_i = \hat{\beta_0} \sum_{i=1}^n X_i  + \hat{\beta_1} \sum_{i=1}^n X_i^2$



---

> Solution to Normal Equations



<div align="center">

$$
\hat{\beta_1} = \frac{\sum_{i=1}^n  (X_i - \bar{X})(Y_i - \bar{Y})}{ \sum_{i=1}^n (X_i - \bar{X})^2} = \frac{S_{xy}}{S_{xx}}
$$

$$
\hat{\beta_0} = (\bar{Y} - \hat{\beta_1} \bar{X})
$$

</div>


<font color='red'>Note, estimators are RVs</font>: Since $Y_i$ is a random variable, and $\hat{\beta_i} = \hat{\beta_i}(Y_1, Y_2, ..., Y_n)$, the estimators will be RVs with their own sampling distribution


---

> Matrix form of OLS

Will see move in "Multiple Linear Regression"

<div align="center">

$$
Q = \argmin_{\beta} ||Y - X\beta||_2^2
$$

$$
\hat{\beta_1} = \hat{\beta} = (X^TX)^{-1} X^T Y
$$

</div>

---

> Gauss Markov Theorem

Under the conditions in the SLR model, the estimators $\beta_0, \beta_1$ are BLUE 


* <font color='red'>Best</font>: $Var(\beta_i)_{min}$ (among all linear estimators)
* <font color='red'>Linear</font>: $b_i = \sum_{i=1}^n k_i Y_i$ where $k_i = f(X_i)$ (wrt RV $Y_i$)
* <font color='red'>Unbiased</font>: $\mathbb{E}[\beta_i] = \beta_i$
* <font color='red'>Estimator</font>: $Cov[\epsilon_i, \epsilon_j] = 0$


---

# <font color='lightblue'> Residuals </font>

> Residuals

This is an estimator of the error term, $e_i = \hat{\epsilon_i}$. It measures <font color='red'> How far is our prediction from the observed value </font>

<div align="center">

$$
e_i = Y_i - \hat{Y_i}
$$

</div>


Properties: 
1. <font color='red'>Centered</font>: $\mathbb{E}[\epsilon_i] = 0$
2. <font color='red'>Homoskedasticity</font>: $Var[\epsilon_i] = \sigma^2$ (constant variance)
3. <font color='red'>Uncorrelated</font>: $Cov[\epsilon_i, \epsilon_j] = 0$
4. <font color='red'>Uncorrelated</font>: $Cov[\epsilon_i, \epsilon_j] = 0$
5. <font color='red'>Passes center of mass</font>: $Cov[\epsilon_i, \epsilon_j] = 0$


Conceptually, $\epsilon_i$ is the error of the true data from the mean predictor 

<font color='purple'> Note, we technically can't estimate the noise since it is a random variable. But `conceptually`, this term tells us how much of our variance is unexplained. </font>

---

> Joint Probability 

This represents the probability of observing two events happening simultaneously. This has two notations as written below (but we will use the later)

<div align="center">

$$
P(A \cap B) = P(A,B)
$$

</div>

Connection to Marginal

--- 


> Conditional Probability

<div align="center">

$$
P(A | B)
$$

</div>

---


> Addition Theorem

We first begin by defining the probability of A or B, $P(A \cup B)$, and the probability of A and B, $P(A \cap B)$, 


<div align="center">

$$
P(A \cup B) = P(A) + P(B) - P(A \cap B)
$$

</div>

--- 


> Multiplication Theorem 


<div align="center">

$$
\prod_{i=1}^n Y_i= Y_1 \cdot Y_2 \cdot ... \cdot Y_n
$$

</div>


---


> Complement 

<div align="center">

$$
\prod_{i=1}^n Y_i= Y_1 \cdot Y_2 \cdot ... \cdot Y_n
$$

</div>


--- 

<img src="https://www.google.com/url?sa=t&source=web&rct=j&url=https%3A%2F%2Ffiveable.me%2Fintroduction-probability%2Funit-1%2Fset-theory-venn-diagrams%2Fstudy-guide%2FjcReY5CAtDqlPMDP&ved=0CBYQjRxqFwoTCJitktLIk5QDFQAAAAAdAAAAABAG&opi=89978449" width=400 align=left>

 (Joint, Marginal, conditional)

# <font color='lightblue'> Maximum Likelihood Estimation (MLE) </font>

> Type I Error 

The probability of rejecting the null hypothesis $H_0$, when it's actually True. $\alpha$ controls this

---

> Type II Error 

The probability of 

---

> Sample Variance

This represents the probability of observing two events happening simultaneously. This has two notations as written below (but we will use the later)

<div align="center">

$$
P(A \cap B) = P(A,B)
$$

</div>

Connection to Marginal

--- 

<font color='purple'> Note: Mean, Sample mean, Variance and Sample variance are all different!!</font>


# <font color='lightblue'> Fisher Information Matrix </font>

> Covariances 

---

> Sample Variance (S^2 = 1/n-1 )

---

> Standard Error (SE = s/n)


---

> Coefficient of Correlation ()

(Covariance, Variances, Coefficient of correlation)

# Further References

1. [Huggingface:](https://huggingface.co/blog/charchits7/understanding-ndcgk-metric) Understnading NDCG@K
2. [Evidently AI:](https://www.evidentlyai.com/ranking-metrics/ndcg-metric) NDCG@K Explained
3. [DigitalSreeni](https://www.youtube.com/watch?v=IMvunY3LrQI&t=406s) NDCG